In [1]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [2]:
# =============================================================================
# BLOCK 2: SYNTHESIZED FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Block 2: Synthesized Feature Engineering ---")
def create_synthesized_features(df_train, df_test):
    # Combine for consistent processing and reset the index
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    # Store the original id for later, as reset_index will remove it
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # --- A) Brute-Force Numerical Interactions ---
    print("Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1','grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] *all_data[NUMS[j]]
    
    # --- B) Date Features ---
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['year'] = all_data['sale_date'].dt.year
    all_data['month'] = all_data['sale_date'].dt.month
    all_data['year_diff'] = all_data['year'] - all_data['year_built']
    
    # --- C) TF-IDF Text Features ---
    print("Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning','join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),max_features=128, binary=True)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        
        # This concat will now work because both have a simple 0-based index
        all_data = pd.concat([all_data, tfidf_df], axis=1)
    
    # --- D) Log transform some of the new interaction features ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            # Add a small constant to avoid log(0)
            all_data[c] = np.log1p(all_data[c].fillna(0))
    
    # --- E) Final Cleanup ---
    print("Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city','sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data.fillna(0, inplace=True)
    
    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train','sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train','sale_price'])
    
    # Restore the original 'id' as the index
    X.index = train_ids
    X_test.index = test_ids
    X_test = X_test[X.columns]
    return X, X_test

# We need to re-run this from the original dataframes
X, X_test = create_synthesized_features(df_train, df_test)
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()


--- Starting Block 2: Synthesized Feature Engineering ---
Creating brute-force numerical interaction features...
Creating TF-IDF features for text columns...
Finalizing feature set...

Synthesized FE complete. Total features: 111


10

In [3]:
# =============================================================================
# BLOCK 3: K-FOLD TRAINING OF MEAN MODEL (NO TUNING)
# =============================================================================
print("\n--- STAGE 1: K-Fold Training of Mean Model ---")
print("# Using pre-tuned, optimal hyperparameters.")
# --- YOUR BEST PARAMETERS FOR THE MEAN MODEL ---
# These are the parameters from your most successful Optuna run.
best_params_mean = {
            'eta': 0.041599605162930035,
            'max_depth': 8,
            'subsample': 0.8211034324219306,
            'colsample_bytree': 0.8683430739702909,
            'lambda': 3.717655605557664,
            'alpha': 2.8186169330124836e-05
            }


# --- K-Fold Training ---
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True,random_state=RANDOM_STATE)
oof_mean_preds = np.zeros(len(X))
test_mean_preds = np.zeros(len(X_test))


# THE FIX: Reload the 'grade' column for stratification as the original df was deleted.
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

# Add the other required XGBoost parameters
final_params_mean = {'objective': 'reg:squarederror', 'eval_metric': 'rmse','tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1,**best_params_mean}
for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f" Mean Model - Fold {fold+1}/{N_SPLITS}...")
    model = xgb.XGBRegressor(**final_params_mean, n_estimators=2500,early_stopping_rounds=100)
    model.fit(X.iloc[train_idx], y_true.iloc[train_idx], eval_set=[(X.iloc[val_idx], y_true.iloc[val_idx])], verbose=False)
    oof_mean_preds[val_idx] = model.predict(X.iloc[val_idx])
    test_mean_preds += model.predict(X_test) / N_SPLITS

    
# --- NEW: CALCULATE AND PRINT FINAL OOF RMSE ---
final_mean_rmse = np.sqrt(mean_squared_error(y_true, oof_mean_preds))
print(f"\n# Mean model K-Fold training complete.")
print(f"# Final OOF RMSE for Mean Model: ${final_mean_rmse:,.2f}")
print("-" * 50)



--- STAGE 1: K-Fold Training of Mean Model ---
# Using pre-tuned, optimal hyperparameters.
 Mean Model - Fold 1/5...
 Mean Model - Fold 2/5...
 Mean Model - Fold 3/5...
 Mean Model - Fold 4/5...
 Mean Model - Fold 5/5...

# Mean model K-Fold training complete.
# Final OOF RMSE for Mean Model: $98,990.27
--------------------------------------------------


In [4]:
# =======================================================================================
#
# BLOCK 4: TRAIN THE CATBOOST MEAN MODEL
#
# =======================================================================================
print("--- STAGE 1: K-Fold Training of CatBoost Mean Model ---")
print("# Using the optimal hyperparameters found previously.")

# The best parameters you found for the CatBoost mean model
best_params_catboost = {
    'iterations': 2329,
    'learning_rate': 0.07106828663102695,
    'depth': 9,
    'l2_leaf_reg': 0.011456642952301135,
    'subsample': 0.901161247633512,
    'random_strength': 0.6138817518014036,
    'bagging_temperature': 0.017313451474763708,
    'random_seed': RANDOM_STATE,
    'verbose': 0
}

# Initialize arrays to store predictions
oof_catboost_preds = np.zeros(len(X))
test_catboost_preds = np.zeros(len(X_test))

# Setup K-Fold splits
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"  Training CatBoost Mean Model - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_true.iloc[train_idx], y_true.iloc[val_idx]
    
    model = cb.CatBoostRegressor(**best_params_catboost)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              early_stopping_rounds=100,
              verbose=0)
    
    oof_catboost_preds[val_idx] = model.predict(X_val)
    test_catboost_preds += model.predict(X_test) / N_SPLITS
    
# --- Evaluate Mean Model Performance ---
final_mean_rmse_cb = np.sqrt(mean_squared_error(y_true, oof_catboost_preds))
print("\n--- CatBoost K-Fold Training Complete & Performance Metrics ---")
print(f"CatBoost Final OOF RMSE: ${final_mean_rmse_cb:,.2f}")

--- STAGE 1: K-Fold Training of CatBoost Mean Model ---
# Using the optimal hyperparameters found previously.
  Training CatBoost Mean Model - Fold 1/5...
  Training CatBoost Mean Model - Fold 2/5...
  Training CatBoost Mean Model - Fold 3/5...
  Training CatBoost Mean Model - Fold 4/5...
  Training CatBoost Mean Model - Fold 5/5...

--- CatBoost K-Fold Training Complete & Performance Metrics ---
CatBoost Final OOF RMSE: $98,246.71


In [5]:
# =======================================================================================
#
# BLOCK 5: STAGE 2 - CREATE ENSEMBLE & TUNE ITS ERROR MODEL
#
# =======================================================================================

# --- PART A: Create and Evaluate the Ensemble Mean ---
print("\n--- STAGE 2, PART A: Creating the Ensemble Mean Predictions ---")

# We will start with a simple 50/50 average of the two models
oof_ensemble_mean = (oof_mean_preds + oof_catboost_preds) / 2
test_ensemble_mean = (test_mean_preds + test_catboost_preds) / 2

# Evaluate the new ensemble mean model's accuracy
ensemble_mean_rmse = np.sqrt(mean_squared_error(y_true, oof_ensemble_mean))
print("\n--- Mean Model RMSE Comparison ---")
print(f"Original XGBoost Mean RMSE: ${final_mean_rmse:,.2f}")
print(f"Tuned CatBoost Mean RMSE  : ${final_mean_rmse_cb:,.2f}")
print(f"Ensemble Mean RMSE        : ${ensemble_mean_rmse:,.2f}  <-- NEW BEST MEAN MODEL!")

# --- PART B: Tune Hyperparameters for the Ensemble's Errors ---
print("\n--- STAGE 2, PART B: Tuning an XGBoost model on the Ensemble's Errors ---")

# 1. Create the NEW error target and feature set
error_target_ensemble = np.abs(y_true - oof_ensemble_mean)
X_for_error_ensemble = X.copy()
X_for_error_ensemble['mean_pred_oof'] = oof_ensemble_mean

# 2. Define Optuna objective function
def create_error_objective(X_features, y_error):
    X_train, X_val, y_train, y_val = train_test_split(X_features, y_error, test_size=0.25, random_state=RANDOM_STATE)
    def objective(trial):
        params = {
            'objective': 'reg:squarederror','eval_metric': 'rmse','tree_method': 'hist',
            'eta': trial.suggest_float('eta', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'lambda': trial.suggest_float('lambda', 1e-8, 10.0, log=True),
            'alpha': trial.suggest_float('alpha', 1e-8, 10.0, log=True),
            'seed': RANDOM_STATE, 'n_jobs': -1
        }
        model = xgb.XGBRegressor(**params, n_estimators=2000, early_stopping_rounds=50)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        preds = model.predict(X_val)
        return np.sqrt(mean_squared_error(y_val, preds))
    return objective

# 3. Run the study
study_error_ensemble = optuna.create_study(direction='minimize')
study_error_ensemble.optimize(create_error_objective(X_for_error_ensemble, error_target_ensemble), n_trials=N_OPTUNA_TRIALS)

# 4. Store the best parameters
best_params_error_ensemble = study_error_ensemble.best_params
print("\n--- Ensemble Error Model Tuning Complete ---")
print(f"Best validation RMSE found for ensemble error model: ${study_error_ensemble.best_value:,.2f}")
print("Best hyperparameters found:", best_params_error_ensemble)

[I 2025-07-16 16:03:51,083] A new study created in memory with name: no-name-e83a36c8-fca9-47b3-9cca-3f583788eff9



--- STAGE 2, PART A: Creating the Ensemble Mean Predictions ---

--- Mean Model RMSE Comparison ---
Original XGBoost Mean RMSE: $98,990.27
Tuned CatBoost Mean RMSE  : $98,246.71
Ensemble Mean RMSE        : $96,994.03  <-- NEW BEST MEAN MODEL!

--- STAGE 2, PART B: Tuning an XGBoost model on the Ensemble's Errors ---


[I 2025-07-16 16:03:54,673] Trial 0 finished with value: 61943.83894198676 and parameters: {'eta': 0.03001216382327973, 'max_depth': 7, 'subsample': 0.7225848590480339, 'colsample_bytree': 0.8474489642310846, 'lambda': 1.5671457496627552e-07, 'alpha': 6.5845956763534664e-06}. Best is trial 0 with value: 61943.83894198676.
[I 2025-07-16 16:03:58,120] Trial 1 finished with value: 62435.2273808038 and parameters: {'eta': 0.0375784553691912, 'max_depth': 4, 'subsample': 0.5743318131033719, 'colsample_bytree': 0.7051462476754886, 'lambda': 0.0011199031499797302, 'alpha': 0.0007156959251342894}. Best is trial 0 with value: 61943.83894198676.
[I 2025-07-16 16:04:05,877] Trial 2 finished with value: 61955.63814919232 and parameters: {'eta': 0.01299481058526311, 'max_depth': 6, 'subsample': 0.8203973128706719, 'colsample_bytree': 0.8437295121192381, 'lambda': 0.009881113026733552, 'alpha': 0.00020917877618704795}. Best is trial 0 with value: 61943.83894198676.
[I 2025-07-16 16:04:09,129] Trial 


--- Ensemble Error Model Tuning Complete ---
Best validation RMSE found for ensemble error model: $61,578.11
Best hyperparameters found: {'eta': 0.02236241073239035, 'max_depth': 8, 'subsample': 0.9092833284980538, 'colsample_bytree': 0.5047646258886728, 'lambda': 0.0191434778750427, 'alpha': 1.1475576923836229e-08}


In [6]:
# =======================================================================================
#
# BLOCK 6: STAGE 3 - K-FOLD TRAIN THE TUNED ENSEMBLE ERROR MODEL
#
# =======================================================================================
print("\n--- STAGE 3: K-Fold Training of the Tuned Ensemble Error Model ---")

# Add fixed parameters to our newly tuned ones
final_params_error_ensemble = best_params_error_ensemble.copy()
final_params_error_ensemble.update({'objective': 'reg:squarederror', 'eval_metric': 'rmse',
                                    'tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1})

# Initialize prediction arrays
oof_error_preds_ensemble = np.zeros(len(X))
test_error_preds_ensemble = np.zeros(len(X_test))
X_test_for_error_ensemble = X_test.copy()
X_test_for_error_ensemble['mean_pred_oof'] = test_ensemble_mean

# Use the same K-Fold splits
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error_ensemble, grade_for_stratify)):
    print(f"  Training Ensemble Error Model - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X_for_error_ensemble.iloc[train_idx], X_for_error_ensemble.iloc[val_idx]
    y_train, y_val = error_target_ensemble.iloc[train_idx], error_target_ensemble.iloc[val_idx]

    model = xgb.XGBRegressor(**final_params_error_ensemble, n_estimators=2000, early_stopping_rounds=100)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    oof_error_preds_ensemble[val_idx] = model.predict(X_val)
    test_error_preds_ensemble += model.predict(X_test_for_error_ensemble) / N_SPLITS

# --- Performance of the New Error Model ---
final_error_rmse_ensemble = np.sqrt(mean_squared_error(error_target_ensemble, oof_error_preds_ensemble))
print(f"\nEnsemble Error Model Final OOF RMSE: ${final_error_rmse_ensemble:,.2f}")


--- STAGE 3: K-Fold Training of the Tuned Ensemble Error Model ---
  Training Ensemble Error Model - Fold 1/5...
  Training Ensemble Error Model - Fold 2/5...
  Training Ensemble Error Model - Fold 3/5...
  Training Ensemble Error Model - Fold 4/5...
  Training Ensemble Error Model - Fold 5/5...

Ensemble Error Model Final OOF RMSE: $61,879.01


In [7]:
# =======================================================================================
#
# BLOCK 7: STAGE 4 - FINAL CALIBRATION AND ENSEMBLE SHOWDOWN
#
# =======================================================================================
print("\n" + "="*60)
print("             THE ENSEMBLE PIPELINE: FINAL SHOWDOWN")
print("="*60)

OLD_BEST_SCORE = 301553.47 # Your champion score from winner_v1_301

oof_error_final_ensemble = np.clip(oof_error_preds_ensemble, 0, None)
best_score_ensemble = float('inf')
best_a_ensemble, best_b_ensemble = 1.0, 1.0

print("Starting final calibration grid search for the ENSEMBLE pipeline...")
for a in np.arange(1.90, 2.31, 0.01):
    for b in np.arange(2.10, 2.51, 0.01):
        low = oof_ensemble_mean - oof_error_final_ensemble * a
        high = oof_ensemble_mean + oof_error_final_ensemble * b
        score = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA)
        if score < best_score_ensemble:
            best_score_ensemble = score
            best_a_ensemble, best_b_ensemble = a, b

print("\n--- FINAL RESULTS ---")
print(f"Original XGBoost Pipeline Final Score: {OLD_BEST_SCORE:,.2f}")
print(f"New ENSEMBLE Pipeline Final Score    : {best_score_ensemble:,.2f}")
print(f"  (Using optimal multipliers a={best_a_ensemble:.2f}, b={best_b_ensemble:.2f})")

if best_score_ensemble < OLD_BEST_SCORE:
    print("\nCONCLUSION: VICTORY! The Ensemble-based pipeline is the new champion!")
else:
    print("\nCONCLUSION: Close! The original XGBoost pipeline remains superior.")


             THE ENSEMBLE PIPELINE: FINAL SHOWDOWN
Starting final calibration grid search for the ENSEMBLE pipeline...

--- FINAL RESULTS ---
Original XGBoost Pipeline Final Score: 301,553.47
New ENSEMBLE Pipeline Final Score    : 298,168.87
  (Using optimal multipliers a=1.95, b=2.17)

CONCLUSION: VICTORY! The Ensemble-based pipeline is the new champion!


In [9]:
# =======================================================================================
#
# BLOCK 8: STAGE 5 - CREATE FINAL SUBMISSION (CORRECTED)
#
# =======================================================================================
if best_score_ensemble < OLD_BEST_SCORE:
    print("\n--- Creating the new champion submission file... ---")
    
    # --- THE FIX IS HERE ---
    # Ignore any previous 'test_ids' variable.
    # Load the correct, original IDs directly from the test.csv file.
    # We use `usecols` to make this very fast and memory-efficient.
    print("Loading original IDs from test.csv...")
    correct_test_ids = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])['id']
    
    # The rest of the prediction calculations are correct as they are already in the right order.
    test_error_final_ensemble = np.clip(test_error_preds_ensemble, 0, None)
    final_lower = test_ensemble_mean - test_error_final_ensemble * best_a_ensemble
    final_upper = test_ensemble_mean + test_error_final_ensemble * best_b_ensemble
    final_upper = np.maximum(final_lower, final_upper)
    
    # Create the submission DataFrame using the CORRECT IDs.
    submission_df = pd.DataFrame({'id': correct_test_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})
    submission_filename = 'submission_ensemble_pipeline_v1_CORRECT_IDS.csv'
    submission_df.to_csv(submission_filename, index=False)

    print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
    display(submission_df.head())
else:
    print("\nNew submission file not created as the score was not an improvement.")


--- Creating the new champion submission file... ---
Loading original IDs from test.csv...

'submission_ensemble_pipeline_v1_CORRECT_IDS.csv' created successfully! Good luck on the leaderboard!


,id,pi_lower,pi_upper
0,200000,834335.404947,1.058136e+06
1,200001,584252.156280,7.980737e+05
2,200002,448058.425868,6.491074e+05
3,200003,286412.686713,4.143234e+05
4,200004,260368.980318,7.972589e+05
